## Example of mid-point arbitrage

In [ ]:
if False:
    import os
    os.environ["QUANTDATA_CACHE_MODE"] = "off" # turns off global cache mode; good for testing, but do not use in general.

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.utils.arbfree
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.arbfree

from datetime import date

from cdxcore.dynaplot import figure
from quantdata.widb.day import collect_raw_day, enrich_day, ExpiryFilter, RawDayFilter
from quantdata.widb.options_filter import filter_options_day, OptionsFilter
from quantdata.widb.arbfree import mk_arb_free_day, MkArbFree
from quantdata.biz_calendar import next_business_day, DAYS_PER_YEAR
from cdxcore.verbose import Context
from IPython.display import display
import numpy as np

def test():
    ticker = "SPX"
    d      = next_business_day( date(2025, 8, 10), count_start=True )

    raw_day_filter = RawDayFilter()
    expiry_filter = ExpiryFilter()
    options_filter = OptionsFilter()
    verbose = Context.all

    raw_day  = collect_raw_day(ticker, d, raw_day_filter=raw_day_filter, verbose=verbose, raise_if_bad_day=False)
    day      = enrich_day(ticker, d, raw_day, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad_day=False)
    #day      = filter_options_day(ticker, d, enr_day, options_filter=options_filter, mk_arb_free=None, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad_day=False)
    
    # mk arb free
    expiries = []
    expiry_id_str = []
    strikes = []
    asks = []
    bids = []

    unids = np.unique( day['expiry_id'] )
    for uid in unids:
        mask = day['expiry_id'] == uid
        expiries.append( day['tte_y'][ mask ].iloc[0] )
        expiry = day['expiry'][ mask ].iloc[0]
        is_am = day['is_am'][ mask ].iloc[0]
        expiry_id_str.append( f"{expiry.date()} {'am' if is_am else 'pm'}" )

        strikes_t = day['pure_strike'][mask].to_numpy()
        ixs = np.argsort(strikes_t)
        strikes.append( strikes_t[ixs] )
        bids.append( day['pure_bid_price'][mask].to_numpy()[ixs] )
        asks.append( day['pure_ask_price'][mask].to_numpy()[ixs] )

    expiries = np.array(expiries)
    T        = len(expiries)

    mkaf     = MkArbFree(smoothness=0.1,surface_expiry_linear=True, timeout_seconds=60.*20)
    af       = mkaf( expiries, strikes, bids, asks, verbose=Context.all )



    with figure(f"Expiries {d}", row_size=4, col_size=6, columns=4) as fig:
        for t in range(T):
            expiry_str = expiry_id_str[t]
            expiry     = expiries[t]
            strikes_t  = strikes[t]
            bids_t     = bids[t]
            asks_t     = asks[t]
            mids_t     = (bids_t + asks_t)/2.
            fits_t     = af.price( expiry, strikes_t, mode="surface" )

            strikesx = np.zeros((len(strikes_t)+2,), dtype=strikes_t.dtype)
            strikesx[1:-1] = strikes_t
            strikesx[0]    = strikes_t[0]*0.5
            strikesx[-1]   = strikes_t[-1]*1.5
            midx       = np.zeros_like(strikesx)
            midx[1:-1] = mids_t
            midx[0]    = 1.- strikesx[0]
            midx[-1]   = 0.
            fitx       = np.zeros_like(strikesx)
            fitx[1:-1] = fits_t
            fitx[0]    = 1.- strikesx[0]
            fitx[-1]   = 0.

            mdCdCK  = np.diff(midx,prepend=1.) / np.diff(strikesx,prepend=0.)
            mp      = np.diff(mdCdCK,append=0.) 
            mratio  = np.sum(mp[mp<0.]) / np.sum(mp)

            fdCdK  = np.diff(fitx,prepend=1.) / np.diff(strikesx,prepend=0.)
            fp     = np.diff(fdCdK,append=0.)
            fratio = np.sum(fp[fp<0.]) / np.sum(fp)

            ax = fig.add_subplot(f"Expiry {expiry_str} TTE {expiry*DAYS_PER_YEAR:.1f}d\nMarket Ratio: {mratio:.2f} Fit Ratio: {fratio:.2f}")
            ax.set_xlabel("Extended strikes")
            ax.plot(strikesx[1:-1], mp[1:-1], ":", label="market p", color="orange")
            ax.plot(strikesx[1:-1], fp[1:-1], ":", label="fit p", color="blue")
            ax.fill_between(strikesx[1:-1], mp[1:-1]*0., np.clip(mp[1:-1], 0., None), color="green", alpha=0.2)
            ax.fill_between(strikesx[1:-1], np.clip(mp[1:-1],min=None,max=0.), mp[1:-1]*0., color="red", alpha=0.2)
            ax.plot(strikesx[1:-1], mp[1:-1]*0., ":", color="purple")
            ax.set_ylabel("p")
            ax.legend(loc="upper left")


_ = test()

00: Expiry #0 1.0d: Generating extended strikes... removed 11 out of 128 strikes to ensure -1 < dC/dK[0] and dC/dK[-1] < 0.using 119 extended strikes based on 128 market strikes.
00: Expiry #0 1.0d: Could not hard fit with linear per-expiry no-arbitrage -- using masked soft fit instead which is outside bid/ask for 4 of 117 options. This took 0.086s.
00: Expiry #0 1.0d: Soft linear per-expiry no-arbitrage fit succeeded within bid/ask. This took 0.031s.
00: Expiry #0 1.0d: fit achived after removing 15 of 128 strikes.
00: Expiry #1 2.0d: Generating extended strikes... removed 9 out of 143 strikes to ensure -1 < dC/dK[0] and dC/dK[-1] < 0.using 136 extended strikes based on 143 market strikes.
00: Expiry #1 2.0d: Soft linear per-expiry no-arbitrage fit succeeded within bid/ask. This took 0.042s.
00: Expiry #2 2.2d: Generating extended strikes... removed 29 out of 291 strikes to ensure -1 < dC/dK[0] and dC/dK[-1] < 0.using 264 extended strikes based on 291 market strikes.
00: Expiry #2 2.2